## Conteos y estado de los indicadores

In [0]:
%sql
SELECT 'bronze.vuelos' AS tabla, COUNT(*) AS registros FROM aerolinea.bronze.vuelos
UNION ALL SELECT 'silver.vuelos', COUNT(*) FROM aerolinea.silver.vuelos
UNION ALL SELECT 'gold.vuelos_enriquecidos', COUNT(*) FROM aerolinea.gold.vuelos_enriquecidos
UNION ALL SELECT 'gold.mantenimientos_enriquecidos', COUNT(*) FROM aerolinea.gold.mantenimientos_enriquecidos
UNION ALL SELECT 'gold.base_ml_mantenimiento', COUNT(*) FROM aerolinea.gold.base_ml_mantenimiento

tabla,registros
bronze.vuelos,100
silver.vuelos,100
gold.vuelos_enriquecidos,100
gold.mantenimientos_enriquecidos,100
gold.base_ml_mantenimiento,100


In [0]:
vuelos = spark.table("aerolinea.gold.vuelos_enriquecidos").count()
assert vuelos == 100, "El conteo no coincide con la instantánea de clase."
assert spark.table("aerolinea.gold.mantenimientos_enriquecidos").count() == 100
assert spark.table("aerolinea.gold.base_ml_mantenimiento").count() == 100

## Reconciliación de conteos y costos

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW control_reconciliacion AS
SELECT 'vuelos_por_grupo' AS regla,
       (SELECT COUNT(*) FROM aerolinea.silver.vuelos) AS esperado,
       (SELECT SUM(total_vuelos) FROM aerolinea.gold.kpi_puntualidad_vuelos) AS obtenido
UNION ALL
SELECT 'mantenimientos_por_grupo',
       (SELECT COUNT(*) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(total_mantenimientos) FROM aerolinea.gold.kpi_costo_mantenimiento)
UNION ALL
SELECT 'vuelos_por_aeronave_mes',
       (SELECT COUNT(*) FROM aerolinea.silver.vuelos),
       (SELECT SUM(total_vuelos) FROM aerolinea.gold.resumen_aeronave_mes)
UNION ALL
SELECT 'mantenimientos_por_aeronave_mes',
       (SELECT COUNT(*) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(total_mantenimientos) FROM aerolinea.gold.resumen_aeronave_mes)
UNION ALL
SELECT 'costo_por_grupo',
       (SELECT SUM(costo_usd) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(costo_total_usd) FROM aerolinea.gold.kpi_costo_mantenimiento)
UNION ALL
SELECT 'costo_por_aeronave_mes',
       (SELECT SUM(costo_usd) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(costo_total_usd) FROM aerolinea.gold.resumen_aeronave_mes)

In [0]:
reconciliacion = spark.table("control_reconciliacion")
display(reconciliacion)
assert reconciliacion.filter("obtenido IS NULL OR esperado <> obtenido").count() == 0, "Un resumen no coincide con el detalle."

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8021117446803956>, line 2
      1 reconciliacion = spark.table("control_reconciliacion")
----> 2 display(reconciliacion)
      3 assert reconciliacion.filter("obtenido IS NULL OR esperado <> obtenido").count() == 0, "Un resumen no coincide con el detalle."

File <command-8021117446803956>, line 1
----> 1 reconciliacion = spark.table("control_reconciliacion")
      2 display(reconciliacion)
      3 assert reconciliacion.filter("obtenido IS NULL OR esperado <> obtenido").count() == 0, "Un resumen no coincide con el detalle."

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `control_reconciliacion` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`].
If you did not qualify the name with a sche

## Revisión de las tasas

In [0]:
%sql
SELECT SUM(total_vuelos) AS total,
       SUM(vuelos_a_tiempo) AS a_tiempo, SUM(vuelos_retrasados) AS retrasados,
       SUM(vuelos_cancelados) AS cancelados, SUM(vuelos_operados) AS operados,
       SUM(rutas_por_revisar) AS rutas_por_revisar,
       ROUND(100.0 * SUM(vuelos_a_tiempo) / NULLIF(SUM(vuelos_operados), 0), 2) AS puntualidad_pct
FROM aerolinea.gold.kpi_puntualidad_vuelos

total,a_tiempo,retrasados,cancelados,operados,rutas_por_revisar,puntualidad_pct
100,40,29,31,69,25,57.97


In [0]:
estados = spark.sql("""
SELECT SUM(total_vuelos) AS total, SUM(vuelos_operados) AS operados,
       SUM(vuelos_a_tiempo) AS a_tiempo, SUM(vuelos_retrasados) AS retrasados,
       SUM(vuelos_cancelados) AS cancelados
FROM aerolinea.gold.kpi_puntualidad_vuelos
""").first()
assert estados["a_tiempo"] + estados["retrasados"] == estados["operados"]
assert estados["operados"] + estados["cancelados"] == estados["total"]

## Consulta, plan, archivos e historial

In [0]:
%sql
SELECT origen_pais, COUNT(*) AS vuelos_operados
FROM aerolinea.gold.vuelos_enriquecidos
WHERE operado
GROUP BY origen_pais

origen_pais,vuelos_operados
Francia,13
Reino Unido,15
España,32
EEUU,9


In [0]:
%sql
EXPLAIN FORMATTED
SELECT origen_pais, COUNT(*) AS operados
FROM aerolinea.gold.vuelos_enriquecidos
WHERE operado
GROUP BY origen_pais

plan
"== Physical Plan == AdaptiveSparkPlan (6) +- == Initial Plan == PhotonResultStage (5) +- PhotonColumnarToRow (4) +- PhotonGroupingAgg (3) +- PhotonProject (2) +- PhotonScan parquet aerolinea.gold.vuelos_enriquecidos (1) (1) PhotonScan parquet aerolinea.gold.vuelos_enriquecidos Output [2]: [origen_pais#12530, operado#12537] DictionaryFilters: [operado#12537] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-uecagwqztr/uc/689d3e21-6de1-4ca2-aa35-f0f97edee87f/f51d390f-e62c-40a8-accf-3f7ac5096c65/__unitystorage/catalogs/765ed4de-5882-4070-aabe-80f81828b268/tables/b626014d-ed88-4062-852e-9e6ac80ca588] ReadSchema: struct RequiredDataFilters: [isnotnull(operado#12537), operado#12537] (2) PhotonProject Input [2]: [origen_pais#12530, operado#12537] Arguments: [origen_pais#12530] (3) PhotonGroupingAgg Input [1]: [origen_pais#12530] Arguments: [origen_pais#12530], [count(1)], [count(1)#12540L], [origen_pais#12530, count(1)#12540L AS operados#12490L], true (4) PhotonColumnarToRow Input [2]: [origen_pais#12530, operados#12490L] (5) PhotonResultStage Input [2]: [origen_pais#12530, operados#12490L] (6) AdaptiveSparkPlan Output [2]: [origen_pais#12530, operados#12490L] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = vuelos_enriquecidos"


In [0]:
%sql
DESCRIBE DETAIL aerolinea.gold.vuelos_enriquecidos

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,d60dddde-a3a6-4419-8cc8-c452ce8098aa,aerolinea.gold.vuelos_enriquecidos,null,,2026-10-24T16:21:11.454Z,2026-10-24T16:21:15.000Z,List(),List(),1,7720,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> b626014d-ed88-4062-852e-9e6ac80ca588, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-f3a1137e-559c-40d3-9083-e137a6d7e0bc, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-196d746c-58d7-4011-ac10-bb83217fdfcd)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE HISTORY aerolinea.gold.vuelos_enriquecidos

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-24T16:21:15.000Z,77414296667716,sofiacastanedaarias4@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""b626014d-ed88-4062-852e-9e6ac80ca588"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f3a1137e-559c-40d3-9083-e137a6d7e0bc"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-196d746c-58d7-4011-ac10-bb83217fdfcd""}, statsOnLoad -> true)",null,List(885625122991227),5bbac74d-c6d9-4a1c-a714-216396e59c16,1024-145503-fsg043nx-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 7720)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
